In [1]:
import sys
import os

root_dir = r"D:/BIN/Project/agentic_chatbot"
os.chdir(root_dir)
sys.path.append(root_dir)

In [2]:
from app.embeddings.embedding_manager import EmbeddingManager, POL_DOCS, PROD_DOCS
from app.retrievers.vector_store import VectorStore
from app.retrievers.retriever import RAGRetriever
from app.intent.base import Intent
from app.intent.intent_router import dectect_intent
from app.graph.state import ChatState
from app.embeddings.build_vector_store import load_all_collections
from app.retrievers.registry import init_retrievers, get_policy_retriever, get_product_retriever
from app.graph.nodes.retrieve_policy import retrieve_policy_node
from app.graph.nodes.retrieve_product import retrieve_product_node
from app.graph.nodes.retrieve_info import retrieve_info_node
from app.intent.intent_router import dectect_intent
from app.graph.graph_builder import build_graph

c:\Users\Asus\AppData\Local\Programs\Python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 2200.52it/s]
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [3]:
embedding_manager = EmbeddingManager()
policy_store = VectorStore("policy")
product_store = VectorStore("product")

if policy_store.collection.count() == 0 or product_store.collection.count() == 0:
    load_all_collections()

policy_retriever = RAGRetriever(policy_store, embedding_manager)
product_retriever = RAGRetriever(product_store, embedding_manager)
init_retrievers(policy_retriever=policy_retriever, product_retriever=product_retriever)

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7923.65it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384
Vector store initialized. Collection: policy
Existing documents in collection: 8
Vector store initialized. Collection: product
Existing documents in collection: 30


In [4]:
pol_retriever =  get_policy_retriever()
prod_retriever = get_product_retriever()

In [5]:
res = prod_retriever.retrieve("so sánh giữa xe Xe có giá 1,999 tỷ đồng, phù hợp cho gia đình cần SUV châu Âu cỡ trung với xe Xe có giá 4,5 tỷ đồng, phù hợp cho người dùng cần SUV cỡ lớn sử dụng đa địa hình", 2, 0.0)
print(res[0])
print(len(res))
print(type(res[0]))

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 29.39it/s]

Generated embeddings with shape: (1, 384)
{'id': 'doc_afe89f3f_23', 'content': 'MG ZS là SUV hạng B sản xuất năm 2022, nhập khẩu từ Trung Quốc, sử dụng động cơ xăng 1.0L tăng áp, hộp số tự động và 5 chỗ ngồi. Xe có giá 538 triệu đồng, phù hợp cho người dùng cần SUV cỡ nhỏ với chi phí hợp lý.', 'metadata': {'fuel': 'Xăng', 'doc_index': 23, 'price_vnd': 538000000, 'origin': 'Trung Quốc', 'content_length': 212, 'brand': 'MG', 'engine': '1.0L Turbo', 'id': 24, 'transmission': 'Tự động', 'seats': 5, 'body_type': 'SUV', 'year': 2022, 'model': 'ZS', 'segment': 'SUV-B'}, 'distance': 0.39779648184776306, 'score': 0.6022035181522369, 'rank': 1}
2
<class 'dict'>


In [8]:
res_1 = prod_retriever.retrieve("Xe zs có giá dưới 600tr và form suv cỡ nhỏ", 10, 0.0)
for p in res_1:
    print(p)

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 153.44it/s]

Generated embeddings with shape: (1, 384)
{'id': 'doc_4c3329b6_3', 'content': 'Honda CR-V là SUV phân khúc C đời 2023, sử dụng động cơ xăng 1.5L tăng áp, hộp số tự động, cấu hình 7 chỗ ngồi. Xe nhập khẩu Thái Lan, giá bán khoảng 1,098 tỷ đồng, phù hợp cho gia đình cần không gian rộng và sử dụng đa mục đích.', 'metadata': {'body_type': 'SUV', 'origin': 'Thái Lan', 'segment': 'SUV-C', 'content_length': 229, 'engine': '1.5L Turbo', 'seats': 7, 'year': 2023, 'doc_index': 3, 'brand': 'Honda', 'fuel': 'Xăng', 'model': 'CR-V', 'transmission': 'Tự động', 'id': 4, 'price_vnd': 1098000000}, 'distance': 0.7002227902412415, 'score': 0.29977720975875854, 'rank': 1}
{'id': 'doc_e546d93b_11', 'content': 'Ford Everest là SUV phân khúc D đời 2023, nhập khẩu Thái Lan, sử dụng động cơ diesel 2.0L, hộp số tự động và cấu hình 7 chỗ ngồi. Xe có giá 1,299 tỷ đồng, phù hợp cho gia đình cần SUV gầm cao và không gian rộng.', 'metadata': {'seats': 7, 'body_type': 'SUV', 'engine': '2.0L Diesel', 'model': 'Everest

In [12]:
res_2 = pol_retriever.retrieve("tôi muốn hỏi về chính sách đổi xe nếu xe có hư hỏng", 5, 0.0)
for p in res_2:
    print(p)

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 99.85it/s]

Generated embeddings with shape: (1, 384)
{'id': 'doc_830f46b5_3', 'content': 'POL-04: Điều khoản bảo hành đối với xe mới. Tất cả các xe ô tô mới do cơ sở phân phối đều được áp dụng chính sách bảo hành chính hãng theo tiêu chuẩn của nhà sản xuất. Thời hạn bảo hành thông thường là 36 tháng hoặc 100.000 km, tùy theo điều kiện nào đến trước. Phạm vi bảo hành bao gồm các lỗi kỹ thuật phát sinh do nhà sản xuất đối với động cơ, hộp số và các hệ thống liên quan. Chính sách bảo hành không áp dụng đối với các hư hỏng do tai nạn, sử dụng sai mục đích, bảo dưỡng không đúng quy định hoặc hao mòn tự nhiên.', 'metadata': {'doc_index': 3, 'source': 'D:\\BIN\\Project\\agentic_chatbot\\data\\raw\\customer_policy.txt', 'content_length': 521, 'code': 'POL-04:'}, 'distance': 0.6296886205673218, 'score': 0.3703113794326782, 'rank': 1}
{'id': 'doc_3ff2bdd7_4', 'content': 'POL-05: Điều khoản bảo hành đối với xe đã qua sử dụng. Đối với xe ô tô đã qua sử dụng, thời hạn và phạm vi bảo hành sẽ được áp dụng theo 